In [ ]:
import numpy as np
from scipy.ndimage import rotate
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Motivational example from the introduction: Rotation of images

# Define inner product
def inner(A,B):
    return np.sum(A*B)

# Parameters
ns = [25, 50]
angles = [10, 30, 45]
order = 3 # use cubic interpolation

results = []
reshape = False

for n in ns:
    for angle in angles:
        print(f'Compute norms for n = {n} and alpha = {angle}')
        # Compute the norm by power iteration:
        def A(x):
            return rotate(x,angle,reshape=reshape, order=order)
        def AT(x):
            return rotate(x,-angle,reshape=reshape, order=order)

        # Power iteration
        v = np.random.randn(n,n)
        xx,yy = np.meshgrid(range(n),range(n))
        v[(xx-n/2)**2+(yy-n/2)**2 > (n/2)**2] = 0
        v /= np.linalg.norm(v,'fro')
        check = 10
        for k in range(100):
            Av = A(v)
            ATAv = AT(Av)
            normATAv = np.linalg.norm(ATAv,'fro')
            normA1 = np.sqrt(np.sum(v*ATAv))
            normA2 = np.linalg.norm(ATAv,'fro')/np.linalg.norm(Av,'fro')
            normA3 = np.sqrt(np.linalg.norm(ATAv,'fro'))
            normA4 = np.linalg.norm(Av,'fro')
            v = ATAv/normATAv
            if k % check == 0:
                print(f'Step {k}. Est (1): {normA1:2.4f} Est (2): {normA2:2.4f}  Est (3): {normA3:2.4f} Est (4): {normA4:2.4f}') 
    
        
        # now our method
        v = np.random.randn(n,n)
        xx,yy = np.meshgrid(range(n),range(n))
        v[(xx-n/2)**2+(yy-n/2)**2 > (n/2)**2] = 0
        v /= np.linalg.norm(v,'fro')
        Av = A(v)
        normAv = np.linalg.norm(Av,'fro')
        #normA = []
        
        for i in range(20000):
            x = np.random.randn(n,n)
            x[(xx-n/2)**2+(yy-n/2)**2 > (n/2)**2] = 0
            x -= inner(x,v)*v
            x /= np.linalg.norm(x,'fro')
            Ax = A(x)
            normAx = np.linalg.norm(Ax,'fro')
            vOld = v.copy()
            a = inner(Ax,Av)
            b = normAx**2 - normAv**2
            tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
            v += tau*x
            v /= np.linalg.norm(v,'fro')
            Av = A(v)
            normAv = np.linalg.norm(Av,'fro')
            #normA.append(normAv)
    
        results.append({
            "n": n,
            "alpha": angle,
            "(1)": normA1,
            "(2)": normA2,
            "(3)": normA3,
            "(4)": normA4,
            "ours": normAv
        })
     
# Create a DataFrame
df = pd.DataFrame(results)
print(df)
   

In [ ]:
# code that prints the tex code for the table
latex_table = df.to_latex(index=False, float_format="%.5f")
print(latex_table)